# Introduction 

## Predicting Bank Loan Defaults

The dataset contains information about credit applicants. Banks, globally, use this kind of dataset and type of informative data to create models to help in deciding on who to accept/refuse for a loan.


# Work plan 🤝🤝🤝🤝🤝

- Analyze and explore data
- Building a Machine Learning Model




# Analyze and Explore DataSet

In [ ]:
#Importing the librarires 

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
plt.style.use("ggplot")  #using style ggplot

%matplotlib inline
from mpl_toolkits.mplot3d import Axes3D

import plotly.graph_objects as go
import plotly.express as px


import scipy
from scipy.stats import chi2
from scipy.stats import chi2_contingency ,pearsonr, spearmanr

from sklearn.preprocessing import StandardScaler , MinMaxScaler

from sklearn.model_selection import train_test_split , GridSearchCV

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression , Perceptron

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn import tree
from sklearn.tree import export_graphviz
from sklearn.metrics import mean_absolute_error ,mean_squared_error, median_absolute_error,confusion_matrix,accuracy_score,r2_score
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import  precision_recall_curve, roc_auc_score, confusion_matrix, accuracy_score, recall_score, precision_score, f1_score,auc, roc_curve, plot_confusion_matrix
from category_encoders import BinaryEncoder
from IPython.display import Image
import matplotlib.pyplot as plt

color = sns.color_palette()
seed = 37

In [ ]:
#Importing the training dataset

df=pd.read_csv("../input/machine-learning/lending_club_loan_dataset.csv")

In [ ]:
# looking the data set
df.head()

In [ ]:
 #print the shape dataset
print("Shape The DataSet ", df.shape )

In [ ]:
#Checking the dtypes of all the columns

df.info()

In [ ]:
#checking null value 
df.isna().sum()

In [ ]:
# Describe value data set
df.describe().round(2)

In [ ]:
# Checking data balance/proportion
loan = df.bad_loan.value_counts().to_frame().rename(columns={"bad_loan":"absolute"})
loan["percent"] = (loan.apply(lambda x: x/x.sum()*100).round(2))
display(loan)


# pie chart
df.bad_loan.value_counts().plot(kind='pie', subplots=True, autopct='%1.2f%%', explode= (0.05, 0.05), startangle=80, legend=True, fontsize=14, figsize=(16,8), textprops={'color':"black"})
plt.legend(["0: paid loan","1: not paid loan"])

- Unbalanced data: target has 80% of default results (value 1) against 20% of loans that ended up by been paid/ non-default (value 0).

In [ ]:
#Make functions
#Describing all the features in the dataset using and abusing graphics. 
#Start by defining a few functions for every chart:
# boxplot, histograms, bar and pie charts, scatterplots, pivot charts, as well as a statistic descriptions.


# General statistics
def stats(x):
    print(f"Variable: {x}")
    print(f"Type of variable: {df[x].dtype}")
    print(f"Total observations: {df[x].shape[0]}")
    detect_null_val = df[x].isnull().values.any()
    if detect_null_val:
        print(f"Missing values: {df[x].isnull().sum()} ({(df[x].isnull().sum() / df[x].isnull().shape[0] *100).round(2)}%)")
    else:
        print(f"Missing values? {df[x].isnull().values.any()}")
        print(f"Unique values: {df[x].nunique()}")
    if df[x].dtype != "O":
        print(f"Min: {int(df[x].min())}")
        print(f"25%: {int(df[x].quantile(q=[.25]).iloc[-1])}")
        print(f"Median: {int(df[x].median())}")
        print(f"75%: {int(df[x].quantile(q=[.75]).iloc[-1])}")
        print(f"Max: {int(df[x].max())}")
        print(f"Mean: {df[x].mean()}")
        print(f"Std dev: {df[x].std()}")
        print(f"Variance: {df[x].var()}")
        print(f"Skewness: {scipy.stats.skew(df[x])}")
        print(f"Kurtosis: {scipy.stats.kurtosis(df[x])}")
        print("")
        
        # Percentiles 1%, 5%, 95% and 99%
        print("Percentiles 1%, 5%, 95%, 99%")
        display(df[x].quantile(q=[.01, .05, .95, .99]))
        print("")
    else:
        print(f"List of unique values: {df[x].unique()}")

In [ ]:
# Variable vs. target chart
def target(x):
    short_0 = df[df.bad_loan == 0].loc[:,x]
    short_1 = df[df.bad_loan == 1].loc[:,x]
    
    a = np.array(short_0)
    b = np.array(short_1)
    
    np.warnings.filterwarnings('ignore')
    
    plt.hist(a, bins=40, density=True, color="g", alpha = 0.6, label='Not-default', align="left")
    plt.hist(b, bins=40, density=True, color="r", alpha = 0.6, label='Default', align="right")
    plt.legend(loc='upper right')
    plt.title(x, fontsize=10, loc="right")
    plt.xlabel('Relative frequency')
    plt.ylabel('Absolute frequency')
    plt.show()
target("grade")


- It is between the upper-grade classes that the highest not-default loans happen.

In [ ]:
# Barh chart
def barh(x):
    df[x].value_counts().plot(kind="barh", figsize=(6,5), fontsize=10, color=sns.color_palette("rocket"), table=False)
    plt.xlabel("Absolute values", fontsize=10)
    plt.xticks(rotation=0, horizontalalignment="center")
    plt.ylabel(x, fontsize=10)
    plt.title(x, fontsize=10, loc="right")
barh("purpose")


In [ ]:
# Bar chart
def bar(x):
    ax = df[x].value_counts().plot(kind="bar", figsize=(6,5), fontsize=10, color=sns.color_palette("rocket"), table=False)
    for p in ax.patches:
        ax.annotate("%.2f" % p.get_height(), (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='center', xytext=(0, 5), textcoords='offset points')
    plt.xlabel(x, fontsize=10)
    plt.xticks(rotation=0, horizontalalignment="center")
    plt.ylabel("Absolute values", fontsize=10)
    plt.title(x, fontsize=10, loc="right")
    
bar("last_major_derog_none")


In [ ]:
targets = df["bad_loan"].unique()

In [ ]:
# Scatter plot
def scatter(x, y):
    
    for target in targets:
        a = df[df["bad_loan"] == target][x]
        b = df[df["bad_loan"] == target][y]
        plt.scatter(a, b, label=f"bad loan: {target}", marker="*")
    
        plt.xlabel(x, fontsize=10)
        plt.ylabel(y, fontsize=10)
        plt.title("abc", fontsize=10, loc="right")
        plt.legend()
        plt.show()
scatter("annual_inc", "revol_util")


- The customers with the lowest annual income are the ones that have more late fees, especially the highest and heavy ones.

In [ ]:
# Pivot_table_mean
def pivot_mean(a, b, c):
    type_pivot_mean = df.pivot_table(
        columns=a,
        index=b,
        values=c, aggfunc=np.mean)
    display(type_pivot_mean)
# Display pivot_table
    type_pivot_mean.sort_values(by=[b], ascending=True).plot(kind="bar", title=(b), figsize=(6,4),fontsize = 12);
# Pivot_table_sum
def pivot_sum(a, b, c):
    type_pivot_sum = df.pivot_table(
        columns=a,
        index=b,
        values=c, aggfunc=np.sum)
    display(type_pivot_sum)
# Display pivot_table
    type_pivot_sum.sort_values(by=[b], ascending=True).plot(kind="bar", title=(b), figsize=(6,4),fontsize = 12);

    
pivot_mean("bad_loan", "purpose", "total_rec_late_fee")


- The late fees occur in a higher frequency amongst loan purposes such as a house, small business, or vacation. On the other hand, wedding and car are the credit purposes with the lowest late fees execution.

# CORRELATIONS
- Heatmap → Pearson method

In [ ]:
mask = np.triu(df.corr(), 1)
plt.figure(figsize=(19, 9))
sns.heatmap(df.corr(), annot=True, vmax=1, vmin=-1, square=True, cmap='BrBG', mask=mask);

- The heatmap shows there are some positive and negative correlations amongst variables.


##Data Wrangling: Cleansing and Feature Selection

## OUTLIERS
- Let’s examine the data and check for any outliers.

### Starting by selecting and filtering numeric and categoric data.

In [ ]:
df_ca = df.select_dtypes(exclude=["int64","float64"]).copy()
df_nu = df.select_dtypes(exclude=["object","category"]).copy()

# Boxplot: Visualizing the numeric data dispersion

fig, axs = plt.subplots(ncols=3, nrows=4, figsize=(16, 8))
index = 0
axs = axs.flatten()
for k,v in df_nu.items():
    sns.boxplot(y=k, data=df_nu, ax=axs[index], orient="h")
    index += 1
    plt.tight_layout(pad=0.4, w_pad=0.5, h_pad=5.0)

In [ ]:
# MISSING VALUES
# Time to detect and eliminate them

for column in df.columns:
    if df[column].isna().sum() != 0:
        missing = df[column].isna().sum()
        portion = (missing / df.shape[0]) * 100
        print(f"'{column}': number of missing values '{missing}' ---> '{portion:.3f}%'")

In [ ]:
# Strategy: Replacing missing values with the mean (average).
df["annual_inc"] = df.annual_inc.fillna(df.annual_inc.mean())
print(f"Fillna done. Anomalies detected: {df.annual_inc.isnull().values.any()}")

In [ ]:
df["home_ownership"] = df.home_ownership.fillna(df.home_ownership.value_counts().index[0])
print(f"Imputation done. Missing values: {df.home_ownership.isnull().sum()}")

In [ ]:
df.dti.value_counts(dropna=False)


In [ ]:
df["dti"] = df.dti.fillna(df.dti.mean())
print(f"Fillna done. Missing values: {df.dti.isnull().values.any()}")

In [ ]:
abs_mv = df.last_major_derog_none.value_counts(dropna=False)
pc_mv = df.last_major_derog_none.value_counts(dropna=False, normalize=True) * 100
pc_mv_df = pd.DataFrame(pc_mv)
pc_mv_df.rename(columns={"last_major_derog_none":"Percent %"}, inplace=True)
abs_pc = pd.concat([abs_mv,pc_mv_df], axis=1)
abs_pc

In [ ]:
df.info()

In [ ]:
df.home_ownership.value_counts(dropna=False)


# FEATURE SELECTION


In [ ]:
df.info()

In [ ]:
df.drop("id", axis=1, inplace=True)


In [ ]:
# Numerical Features and Categorical/Binary Target
# Selecting numeric variables only:

data_nu = df.select_dtypes(exclude=["object","category"]).copy()
#Creating subsets:

X = data_nu.drop(["bad_loan"], axis= "columns")
y = data_nu.bad_loan

In [ ]:
#Categorical Features and Categorical/Binary Target
#Selecting categoric variables only:

Xcat = df.select_dtypes(exclude=['int64','float64']).copy()
#Creating subsets:

Xcat['target'] = df.bad_loan
Xcat.dropna(how="any", inplace=True)
ycat = Xcat.target
Xcat.drop("target", axis=1, inplace=True)

In [ ]:
# Chi-square test for independence:
for col in Xcat.columns:
    table = pd.crosstab(Xcat[col], ycat)
    print()
    display(table)
    _, pval, _, expected_table = scipy.stats.chi2_contingency(table)
    print(f"p-value: {pval:.25f}")

In [ ]:
#Variable: ‘grade’

df["grade"] = df.grade.map({"A":7, "B":6, "C":5, "D":4, "E":3, "F":2, "G":1})

In [ ]:
#Variables: ‘term’, ‘home_ownership’, ‘purpose’
#One Hot Encoding and Binary Encoding will be both displayed so we can chose the best to apply.

df_term = df.term
df_home = df.home_ownership
df_purp = df.purpose
#term
t_ohe = pd.get_dummies(df_term)
bin_enc_term = BinaryEncoder()
t_bin = bin_enc_term.fit_transform(df_term)
#home_ownsership
h_ohe = pd.get_dummies(df_home)
bin_enc_home = BinaryEncoder()
h_bin = bin_enc_home.fit_transform(df_home)
#purpose
p_ohe = pd.get_dummies(df_purp)
bin_enc_purp = BinaryEncoder()
p_bin = bin_enc_purp.fit_transform(df_purp)

In [ ]:
#One Hot Encoding (OHE)
df = pd.get_dummies(df, columns=["term","home_ownership"])

In [ ]:
#Binary Encoding
bin_enc_purp = BinaryEncoder()
data_bin = bin_enc_purp.fit_transform(df.purpose)

In [ ]:
df.head()

In [ ]:
df.drop(columns=["purpose"],inplace=True)

In [ ]:
#Defined X value and y value , and split the data train

X = df.drop(columns="bad_loan")           
y = df["bad_loan"]    # y = quality

# split the data train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print("X Train : ", X_train.shape)
print("X Test  : ", X_test.shape)
print("Y Train : ", y_train.shape)
print("Y Test  : ", y_test.shape)

# Notes

## Just needed choice model ,  I will do an update my notebook - soon 

- Thank for reading my analysis and my regression.

- If you any questions or advice me please write in the comment .

- If anyone has a model with a higher percentage, please tell me


# Vote

- If you liked my work vote me ,

# The End